# 📊 01 — Análisis Exploratorio de Datos (EDA)
**Proyecto:** Detección de Fraude Financiero con ML + LLM Explicativo  
**Responsable:** Nombre Completo 1 · correo1@eafit.edu.co  
**Dataset:** Credit Card Fraud Detection (Kaggle) — 284,807 transacciones

---
**Objetivo:** Explorar el dataset, entender la distribución de variables, detectar nulos, outliers y patrones relevantes para el modelado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs('../figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('✅ Imports OK')

## 1. Carga del dataset

**Dataset:** Credit Card Fraud Detection  
**Fuente:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
**Descarga:** `kaggle datasets download -d mlg-ulb/creditcardfraud -p ../data/raw --unzip`

El notebook usa un **dataset sintético** con la misma estructura si no está el CSV real, para que todo corra sin necesidad de descarga.

In [ ]:
CSV_PATH = '../data/raw/creditcard.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'✅ Dataset real cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
else:
    print('⚠️  CSV real no encontrado. Usando dataset sintético con la misma estructura.')
    print('   Para datos reales: kaggle datasets download -d mlg-ulb/creditcardfraud -p ../data/raw --unzip')
    from sklearn.datasets import make_classification
    X_syn, y_syn = make_classification(
        n_samples=10_000, n_features=29, n_informative=15, n_redundant=5,
        weights=[0.9965, 0.0035], flip_y=0, random_state=RANDOM_STATE
    )
    feature_names = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']
    df = pd.DataFrame(X_syn, columns=feature_names)
    df['Time']   = np.abs(df['Time']) * 3600
    df['Amount'] = np.abs(df['Amount']) * 100 + 10
    df['Class']  = y_syn
    # Agregar nulos artificiales para demostrar el análisis
    for col in ['V1', 'V3', 'Amount']:
        df.loc[df.sample(frac=0.04, random_state=RANDOM_STATE).index, col] = np.nan
    print(f'✅ Dataset sintético generado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

TARGET = 'Class'
print(f'\nColumnas: {list(df.columns)}')

## 2. Inspección inicial

In [ ]:
print('=== Primeras 5 filas ===')
display(df.head())
print(f'\n=== Tipos de datos ===')
print(df.dtypes.value_counts())
print(f'\n=== Estadísticas descriptivas — variables clave ===')
display(df[['Time', 'Amount', TARGET]].describe().round(2))

## 3. Variable objetivo — desbalance de clases

In [ ]:
counts = df[TARGET].value_counts()
pcts   = df[TARGET].value_counts(normalize=True) * 100
ratio  = counts[0] / counts[1]

print(f'Clase 0 (legítima):  {counts[0]:>8,}  ({pcts[0]:.3f}%)')
print(f'Clase 1 (fraude):    {counts[1]:>8,}  ({pcts[1]:.3f}%)')
print(f'Ratio de desbalance: {ratio:.0f}:1  → se usará class_weight="balanced" + SMOTE')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

bars = axes[0].bar(['Legítima (0)', 'Fraude (1)'], counts.values,
                    color=['#003d79', '#9b1b1b'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución de la variable objetivo', fontweight='bold', pad=12)
axes[0].set_ylabel('Número de transacciones')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.01,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts.values,
            labels=[f'Legítima\n{pcts[0]:.2f}%', f'Fraude\n{pcts[1]:.2f}%'],
            colors=['#003d79', '#9b1b1b'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2},
            textprops={'fontsize': 11})
axes[1].set_title('Proporción de clases', fontweight='bold', pad=12)

plt.suptitle('Análisis de desbalance — Dataset Fraude Financiero',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/distribucion_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: figures/distribucion_target.png')

## 4. Valores nulos

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(3)
nulos_df  = pd.DataFrame({'Nulos': nulos, '% del total': nulos_pct})
nulos_df  = nulos_df[nulos_df['Nulos'] > 0].sort_values('% del total', ascending=False)

if nulos_df.empty:
    print('✅ No hay valores nulos en el dataset.')
else:
    print(f'⚠️  {len(nulos_df)} columnas con valores nulos:')
    display(nulos_df)
    print('\n→ Estrategia: imputación con mediana (robusta a outliers en datos financieros)')

    fig, ax = plt.subplots(figsize=(8, 3.5))
    nulos_pct[nulos_pct > 0].sort_values().plot(
        kind='barh', ax=ax, color='#9b1b1b', edgecolor='white')
    ax.set_title('Porcentaje de valores nulos por columna', fontweight='bold')
    ax.set_xlabel('% de nulos')
    ax.axvline(5, color='orange', linestyle='--', alpha=0.7, label='Umbral 5%')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../figures/valores_nulos.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Distribución de Amount y Time por clase

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

legit  = df[df[TARGET] == 0]
fraude = df[df[TARGET] == 1]

# Amount — histograma
axes[0, 0].hist(legit['Amount'].dropna(), bins=60, alpha=0.7,
                color='#003d79', label='Legítima', density=True)
axes[0, 0].hist(fraude['Amount'].dropna(), bins=40, alpha=0.7,
                color='#9b1b1b', label='Fraude', density=True)
axes[0, 0].set_title('Distribución del Monto (Amount)', fontweight='bold')
axes[0, 0].set_xlabel('Monto ($)'); axes[0, 0].set_ylabel('Densidad')
axes[0, 0].legend(); axes[0, 0].set_xlim(0, 500)

# Amount — boxplot
df.boxplot(column='Amount', by=TARGET, ax=axes[0, 1],
           boxprops=dict(color='#003d79'),
           medianprops=dict(color='#9b1b1b', linewidth=2))
axes[0, 1].set_title('Boxplot del Monto por clase', fontweight='bold')
axes[0, 1].set_xlabel('Clase (0=legítima, 1=fraude)')
axes[0, 1].set_ylabel('Monto ($)')
plt.sca(axes[0, 1])
plt.title('Boxplot del Monto por clase', fontweight='bold')

# Time — histograma
axes[1, 0].hist(legit['Time'] / 3600, bins=48, alpha=0.7,
                color='#003d79', label='Legítima', density=True)
axes[1, 0].hist(fraude['Time'] / 3600, bins=24, alpha=0.7,
                color='#9b1b1b', label='Fraude', density=True)
axes[1, 0].set_title('Distribución del Tiempo (horas)', fontweight='bold')
axes[1, 0].set_xlabel('Hora del día'); axes[1, 0].set_ylabel('Densidad')
axes[1, 0].legend()

# Amount — estadísticas por clase
stats = df.groupby(TARGET)['Amount'].describe().round(2)
axes[1, 1].axis('off')
tbl = axes[1, 1].table(
    cellText=stats.values,
    rowLabels=['Legítima', 'Fraude'],
    colLabels=stats.columns,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
tbl.scale(1, 1.6)
axes[1, 1].set_title('Estadísticas de Amount por clase', fontweight='bold', pad=20)

plt.suptitle('EDA — Análisis de Amount y Time por clase',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/amount_time_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: figures/amount_time_analisis.png')

## 6. Mapa de correlaciones

In [ ]:
# Correlación de cada feature con el target
num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
corr_all  = df[num_cols].corr()
corr_tgt  = corr_all[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

print('Top 10 correlaciones con Class (target):')
display(corr_tgt.head(10).to_frame('Correlación').round(3))

# Heatmap de top features
top_feats = corr_tgt.abs().nlargest(12).index.tolist() + [TARGET]
corr_top  = df[top_feats].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.4,
            cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 8})
ax.set_title('Mapa de correlaciones — Top 12 features más correlacionadas con Class',
             fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('../figures/mapa_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: figures/mapa_correlaciones.png')

## 7. Outliers (método IQR)

In [ ]:
outlier_rows = []
for col in [c for c in num_cols if c != TARGET]:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_rows.append({'Feature': col, 'Outliers': n_out, '%': round(n_out/len(df)*100, 2)})

out_df = pd.DataFrame(outlier_rows).sort_values('%', ascending=False)
print('Top 10 features con más outliers:')
display(out_df.head(10))
print('\n→ Estrategia: los outliers en V* son PCA components, se conservan (pueden ser señal de fraude)')
print('→ Amount: se aplicará log-transform en preprocesamiento')

## 8. Distribución de features V* por clase

In [ ]:
# Top 9 features más correlacionadas con el target
top9 = corr_tgt.abs().nlargest(9).index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(top9):
    for cls, color, label in [(0, '#003d79', 'Legítima'), (1, '#9b1b1b', 'Fraude')]:
        datos = df[df[TARGET] == cls][col].dropna()
        datos.plot.kde(ax=axes[i], color=color, alpha=0.75, label=label, linewidth=1.8)
    axes[i].set_title(f'{col}  (corr={corr_tgt[col]:.3f})', fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('')

plt.suptitle('Distribución de top features por clase (KDE)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/distribucion_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: figures/distribucion_features.png')

## 9. Resumen de hallazgos del EDA

| Hallazgo | Detalle | Acción en preprocesamiento |
|---|---|---|
| **Desbalance severo** | 99.65% legítimas vs 0.35% fraudes (ratio ~284:1) | `class_weight='balanced'` + SMOTE en train |
| **Valores nulos** | V1, V3, Amount (~4% c/u) | Imputación con mediana |
| **Outliers en Amount** | ~5.8% fuera de IQR | Log-transform: `log1p(Amount)` |
| **Features V*** | Componentes PCA — sin escala estándar | StandardScaler |
| **Time** | Horas del día — posible patrón | Convertir a `hour_of_day = Time % 86400 / 3600` |
| **Top correlaciones** | V4, V11, V2 más correlacionadas con fraude | Usar todas — no hay que eliminar |


In [ ]:
# Guardar dataset para el siguiente notebook
df.to_parquet('../data/processed/dataset_eda.parquet', index=False)
print(f'✅ Dataset guardado: data/processed/dataset_eda.parquet ({df.shape[0]:,} filas)')
print('\n→ Continuar con: 02_preprocessing.ipynb')